# 01 — Simulación reproducible de ferries

## Pregunta de negocio
¿Cómo construir una base sintética que permita probar una torre de control sin utilizar datos confidenciales?

El notebook genera servicios, asigna buques físicos, construye rotaciones y valida que no existan solapamientos programados.

## Decisiones de modelado

- Una fila representa un servicio programado.
- La red incluye ocho rutas ficticias.
- La flota contiene 30 activos individuales.
- Cada activo necesita tiempo de escala.
- El retraso puede propagarse a la siguiente rotación.
- La semilla fija garantiza reproducibilidad.

In [1]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.simulator import FLEET, ROUTES, SimulationConfig, simulate_operations

In [2]:
config = SimulationConfig.from_dict(json.loads((ROOT / 'config/simulation.json').read_text(encoding='utf-8')))
display(ROUTES)
display(FLEET.groupby('vessel_type').agg(buques=('vessel_id','count'), capacidad=('capacity_pax','first'), escala_min=('turnaround_min','first')))

,route_id,origin,destination,distance_nm,duration_min,fare_factor,delay_factor
0,DEN-IBZ,DEN,IBZ,57,135,1.05,0.95
1,DEN-FOR,DEN,FOR,61,150,1.00,0.90
2,DEN-PMI,DEN,PMI,125,300,1.15,1.05
3,VAL-IBZ,VAL,IBZ,95,315,1.18,1.12
4,VAL-PMI,VAL,PMI,140,450,1.20,1.15
5,BCN-PMI,BCN,PMI,112,450,1.25,1.18
6,BCN-IBZ,BCN,IBZ,150,540,1.28,1.22
7,MAH-BCN,MAH,BCN,130,420,1.20,1.10


,buques,capacidad,escala_min
vessel_type,,,
ECO,10,850,45
FAST,10,450,35
RO_PAX,10,1200,60


In [3]:
df = simulate_operations(config)
print(f'Servicios: {len(df):,}')
print(f'Buques: {df.vessel_id.nunique()}')
df.head()

Servicios: 12,000
Buques: 30


,operation_id,route_id,origin,destination,vessel_id,vessel_type,rotation_sequence,previous_operation_id,turnaround_min,scheduled_gap_min,...,crew_cost_eur,port_fees_eur,maintenance_cost_eur,delay_cost_eur,cancellation_cost_eur,total_cost_eur,margin_eur,month,weekday,hour
0,OP000001,DEN-PMI,DEN,PMI,ROPAX-09,RO_PAX,40,OP005229,60,795.0,...,3550.0,7774.40,0.0,89.20,0.0,14026.11,9621.25,2,Sunday,13
1,OP000002,DEN-PMI,DEN,PMI,ECO-02,ECO,362,OP001870,45,465.0,...,3100.0,6520.09,0.0,170.81,0.0,12047.15,35601.80,10,Friday,8
2,OP000003,MAH-BCN,MAH,BCN,ECO-09,ECO,301,OP006135,45,915.0,...,4340.0,6541.97,0.0,51.15,0.0,13279.62,32022.19,8,Wednesday,15
3,OP000004,DEN-IBZ,DEN,IBZ,FAST-10,FAST,147,OP001992,35,570.0,...,1395.0,4303.24,0.0,75.04,0.0,7127.03,18197.06,6,Tuesday,9
4,OP000005,DEN-IBZ,DEN,IBZ,ECO-03,ECO,200,OP001496,45,420.0,...,1395.0,6483.23,0.0,34.76,0.0,8941.84,24591.37,6,Sunday,7


## Controles de integridad

In [4]:
checks = {
    'operation_id único': df.operation_id.is_unique,
    'ocupación 0–1': df.occupancy.between(0,1).all(),
    'pasajeros ≤ capacidad': (df.passengers <= df.capacity_pax).all(),
    'cancelados sin hora real': df.loc[df.canceled, 'actual_start'].isna().all(),
    'sin conflictos programados': not df.schedule_conflict_flag.any(),
}
pd.Series(checks, name='resultado')

operation_id único            True
ocupación 0–1                 True
pasajeros ≤ capacidad         True
cancelados sin hora real      True
sin conflictos programados    True
Name: resultado, dtype: bool

In [5]:
completed = df.loc[~df.canceled]
fig, axes = plt.subplots(1,2,figsize=(12,4))
completed.delay_min.clip(upper=60).hist(bins=30, ax=axes[0])
axes[0].set(title='Distribución del retraso', xlabel='Minutos')
completed.occupancy.hist(bins=25, ax=axes[1])
axes[1].set(title='Distribución de ocupación', xlabel='Ocupación')
plt.tight_layout(); plt.show()

In [6]:
summary = pd.Series({
    'servicios afectados': int((completed.propagated_delay_min > 0).sum()),
    'porcentaje afectado': 100 * (completed.propagated_delay_min > 0).mean(),
    'minutos propagados': completed.propagated_delay_min.sum(),
})
summary.round(2)

servicios afectados     944.00
porcentaje afectado       7.94
minutos propagados     8914.80
dtype: float64

## Conclusión
La base representa activos físicos y secuencias operativas, no servicios independientes sin relación.

In [7]:
out = ROOT / 'data/raw/ferry_operations_synthetic.csv'
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
out

PosixPath('/home/runner/work/Levante-Ferries-Operaciones/Levante-Ferries-Operaciones/data/raw/ferry_operations_synthetic.csv')